# AI IT Helpdesk Agent: run on Google Colab
Run the cells **top to bottom** (click ▶ on each, wait for it to finish before the next).
Tip: *Runtime > Change runtime type > T4 GPU* makes embeddings and Llama faster (optional).

## Step 1: Upload the project zip

In [ ]:
# A file picker appears: choose the project zip (any zip name works)
from google.colab import files
import os, glob
files.upload()
zips = glob.glob('*.zip')
assert zips, 'No zip uploaded. Run this cell again and choose the project zip'
!unzip -o -q "{zips[0]}"
apps = [p for p in glob.glob('**/app.py', recursive=True)]
assert apps, 'app.py not found inside the zip'
os.chdir(os.path.dirname(os.path.abspath(apps[0])))   # works whatever the folder is called
!pip install -q -r requirements.txt
print('Ready in', os.getcwd())

## Step 2: Install Ollama and download Llama 3.2 1B (takes 2-3 minutes)

In [ ]:
!apt-get install -y -q zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
!ollama pull llama3.2:1b

## Step 3: Measure retrieval quality
Compares TF-IDF vs embeddings vs hybrid and prints the best confidence threshold for each.
Read the **SUMMARY** at the bottom, then copy the `embedding` and `hybrid` thresholds into `DEFAULT_THRESHOLDS` in `engine.py` (left file panel > double-click engine.py > edit > Ctrl+S).

In [ ]:
!python evaluate.py

## Step 4: Start the app and get a public link

In [ ]:
import subprocess, time, re, os
!pkill -f streamlit
!pkill -f cloudflared
if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
subprocess.Popen('streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1', shell=True)
time.sleep(10)
subprocess.Popen('./cloudflared tunnel --url http://localhost:8501 > tunnel.log 2>&1', shell=True)
url = None
for _ in range(30):
    time.sleep(2)
    found = re.findall(r'https://[-a-z0-9]+\.trycloudflare\.com', open('tunnel.log').read())
    if found:
        url = found[0]; break
print('Open this link:', url if url else 'not ready yet; run this cell again or check tunnel.log')

## Troubleshooting
- Link does not open: wait 20 seconds and refresh, or re-run Step 4.
- Sidebar says *Ollama offline*: re-run Step 2.
- Sidebar says *Embeddings unavailable*: re-run Step 1's `pip install` line, then restart Step 4.
- The app stops when this Colab session ends; re-run Steps 2 and 4 before a demo.